# Assignment 2 - Analitik Kualitas Wi-Fi

**Nama:** Rico Shandika Jovial Agista  
**NIM:** 202410370110433  
**Tema:** Perbandingan kualitas Wi-Fi 2.4 GHz dan 5 GHz pada Mobile Legends

## 1. Struktur data utama

- `menu` menggunakan **tuple** karena daftar menu bersifat tetap.
- `akun` menggunakan **dictionary** untuk pasangan username dan password.
- `profil` menggunakan **nested dictionary** untuk menyimpan detail pengguna.
- `log` menggunakan **list of dictionaries** karena jumlah catatan dapat bertambah dan setiap catatan memiliki beberapa atribut.

In [24]:
menu = ("Register", "Login", "Catat", "Riwayat", "Kalibrasi", "Hapus", "Logout", "Keluar")

akun = {"Rico": "123", "Shandika": "456", "Jovial": "789"}
user_ids = {"Rico": 1, "Shandika": 2, "Jovial": 3}

profil = {
    "Rico": {"nama": "Rico", "phone": "0812", "role": "Analyst"},
    "Shandika": {"nama": "Shandika", "phone": "0821", "role": "Player"},
    "Jovial": {"nama": "Jovial", "phone": "0857", "role": "Coach"},
}

log = [
    {"id": 1, "no": 1, "band": "2.4GHz", "ping": 45, "loss": 1, "stable": True},
    {"id": 2, "no": 1, "band": "5GHz", "ping": 18, "loss": 0, "stable": True},
    {"id": 3, "no": 1, "band": "2.4GHz", "ping": 82, "loss": 5, "stable": False},
]

print("Menu:", menu)
print("Jumlah akun:", len(akun))
print("Jumlah log awal:", len(log))

Menu: ('Register', 'Login', 'Catat', 'Riwayat', 'Kalibrasi', 'Hapus', 'Logout', 'Keluar')
Jumlah akun: 3
Jumlah log awal: 3


## 2. Validasi input

Fungsi `minta` akan terus meminta input sampai data valid. Parameter `t` menentukan tipe data yang diharapkan, misalnya `str` atau `int`.

In [25]:
def minta(p, t=str):
    while True:
        try:
            v = input(p).strip()
            if t is str and not v:
                raise ValueError
            return t(v)
        except ValueError:
            print("  [!] Input tidak valid.")

## 3. Menu berdasarkan status pengguna

Slicing digunakan untuk mengambil bagian tertentu dari tuple menu. Guest hanya dapat melihat Register dan Login, sedangkan pengguna yang sudah login dapat mengakses fitur pencatatan.

In [26]:
def tampil(u=None):
    pilihan = menu[0:2] + (menu[7],) if u is None else menu[2:7] + (menu[7],)
    for nomor, nama_menu in enumerate(pilihan, 1):
        print(f"  {nomor}. {nama_menu}")
    return pilihan


print("Menu guest:")
tampil()
print("\nMenu pengguna login:")
tampil("Rico")

Menu guest:
  1. Register
  2. Login
  3. Keluar

Menu pengguna login:
  1. Catat
  2. Riwayat
  3. Kalibrasi
  4. Hapus
  5. Logout
  6. Keluar


('Catat', 'Riwayat', 'Kalibrasi', 'Hapus', 'Logout', 'Keluar')

## 4. Register dan login

Register menambahkan akun dan profil baru. Login memeriksa apakah username ada dan password sesuai.

In [27]:
def register():
    u = minta("  Username : ")
    if u in akun:
        print("  [!] Sudah terdaftar.")
        return
    akun[u] = minta("  Password : ")
    user_ids[u] = max(user_ids.values()) + 1
    profil[u] = {
        "nama": minta("  Nama : "),
        "phone": minta("  HP : "),
        "role": minta("  Role : "),
    }
    print("  [OK] Terdaftar.")


def login():
    u = minta("  Username : ")
    if u in akun and akun[u] == minta("  Password : "):
        print(f"  [OK] Halo {profil[u]['nama']}")
        return user_ids[u]
    print("  [!] Gagal login.")


print("Fungsi register dan login siap digunakan.")

Fungsi register dan login siap digunakan.


## 5. Mencatat dan melihat riwayat

Setiap operasi dibatasi dengan `id == uid`, sehingga pengguna hanya mengakses catatannya sendiri. Nilai `stable` dihitung otomatis berdasarkan ping dan loss.

In [28]:
def catat(uid):
    b = minta("  Band (2.4GHz/5GHz): ")
    if b not in ("2.4GHz", "5GHz"):
        print("  [!] Band salah.")
        return
    p = minta("  Ping : ", int)
    l = minta("  Loss : ", int)
    no = 1 + sum(1 for x in log if x["id"] == uid)
    log.append({
        "id": uid,
        "no": no,
        "band": b,
        "ping": p,
        "loss": l,
        "stable": p < 50 and l < 2,
    })
    print(f"  [OK] Log #{no} disimpan.")


def riwayat(uid):
    data = [x for x in log if x["id"] == uid]
    if not data:
        print("  (kosong)")
        return
    for x in data:
        print(f"  #{x['no']} {x['band']} ping={x['ping']} loss={x['loss']} stable={x['stable']}")


print("Riwayat Rico:")
riwayat(1)

Riwayat Rico:
  #1 2.4GHz ping=45 loss=1 stable=True


## 6. Kalibrasi dan hapus log

`kalibrasi` mengubah nilai ping pada log tertentu. `hapus` menghapus log tertentu dengan `pop`. Keduanya hanya bekerja pada log milik pengguna yang sedang login.

In [29]:
def kalibrasi(uid):
    no = minta("  Nomor log : ", int)
    for x in log:
        if x["id"] == uid and x["no"] == no:
            x["ping"] = minta("  Ping baru : ", int)
            x["stable"] = x["ping"] < 50 and x["loss"] < 2
            print("  [OK] Kalibrasi berhasil.")
            return
    print("  [!] Tidak ditemukan.")


def hapus(uid):
    no = minta("  Nomor log : ", int)
    for i, x in enumerate(log):
        if x["id"] == uid and x["no"] == no:
            log.pop(i)
            print("  [OK] Dihapus.")
            return
    print("  [!] Tidak ditemukan.")

## 7. Demo fungsi tanpa input interaktif

Cell ini hanya menampilkan data yang sudah tersedia. Fungsi yang membutuhkan `input()` tidak dipanggil saat **Run All**, sehingga notebook tidak berhenti menunggu masukan.

In [30]:
print("Profil pengguna:")
for username, data in profil.items():
    print(f"- {username}: {data['nama']} ({data['role']})")

print("\nRiwayat Shandika:")
riwayat(2)

print("\nRiwayat Jovial:")
riwayat(3)

Profil pengguna:
- Rico: Rico (Analyst)
- Shandika: Shandika (Player)
- Jovial: Jovial (Coach)

Riwayat Shandika:
  #1 5GHz ping=18 loss=0 stable=True

Riwayat Jovial:
  #1 2.4GHz ping=82 loss=5 stable=False


## 8. Program utama interaktif

Program utama dibungkus dalam fungsi `main()` agar tidak langsung menjalankan loop ketika notebook di-Run All. Jalankan cell berikut secara manual jika ingin mencoba menu interaktif.

In [31]:
def main():
    u = None
    while True:
        pilihan_menu = tampil(u)
        pilihan = minta("  Pilih : ", int)
        if not 1 <= pilihan <= len(pilihan_menu):
            continue
        aksi = pilihan_menu[pilihan - 1]
        if aksi == "Register":
            register()
        elif aksi == "Login":
            u = login() or u
        elif aksi == "Catat":
            catat(u)
        elif aksi == "Riwayat":
            riwayat(u)
        elif aksi == "Kalibrasi":
            kalibrasi(u)
        elif aksi == "Hapus":
            hapus(u)
        elif aksi == "Logout":
            print("  [OK] Logout.")
            u = None
        elif aksi == "Keluar":
            print("Program selesai.")
            break

## Ringkasan konsep

- Tuple digunakan untuk menu yang tetap.
- Dictionary digunakan untuk autentikasi dan profil.
- Nested dictionary menyimpan data profil yang memiliki beberapa atribut.
- List of dictionaries cocok untuk log yang jumlahnya dinamis.
- Slicing membedakan menu guest dan menu pengguna login.
- `try-except` melakukan validasi input.
- CRUD dibatasi berdasarkan identitas pengguna yang sedang login.